# Подготовка benchmark-предсказаний

Финальное решение объединяет четыре retrieval-канала:

- Local E5;
- Local TF-IDF;
- Nearby E5;
- Global E5.

Кандидаты агрегируются с помощью Logistic Regression,
выбранной по Recall@50 на локальной отложенной выборке.

In [1]:
import os
import sys
from pathlib import Path

import faiss
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import BallTree, NearestNeighbors

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT))

from src.geo import build_location_centers
from src.retrieval import (
    encode_items,
    encode_queries,
    load_semantic_model,
)
from src.text_features import (
    build_lexical_item_text,
    build_query_text,
    build_semantic_item_text,
)

DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "benchmark"

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

/Users/arina/Documents/Projects/RecSys_candidate_generation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_parquet(
    DATA_DIR / "train.parquet"
)

benchmark_queries = pd.read_parquet(
    DATA_DIR / "benchmark_queries.parquet"
).reset_index(drop=True)

benchmark_items = pd.read_parquet(
    DATA_DIR / "benchmark_items.parquet"
).reset_index(drop=True)

In [3]:
assert benchmark_queries["query_id"].is_unique
assert benchmark_items["item_id"].is_unique

assert benchmark_queries["query_id"].str.len().eq(16).all()
assert benchmark_items["item_id"].str.len().eq(16).all()

benchmark_queries.shape, benchmark_items.shape

((2452, 6), (189212, 14))

In [4]:
query_texts = build_query_text(
    benchmark_queries
)

lexical_item_texts = build_lexical_item_text(
    benchmark_items
)

semantic_item_texts = build_semantic_item_text(
    benchmark_items
)

len(query_texts), len(lexical_item_texts), len(semantic_item_texts)

(2452, 189212, 189212)

In [5]:
location_reference_items = pd.concat(
    [
        train[
            [
                "item_location_id",
                "item_latitude",
                "item_longitude",
            ]
        ],
        benchmark_items[
            [
                "item_location_id",
                "item_latitude",
                "item_longitude",
            ]
        ],
    ],
    ignore_index=True,
).drop_duplicates()

location_centers = build_location_centers(
    location_reference_items
)

In [6]:
query_location_centers = (
    location_centers
    .reset_index()
    .rename(
        columns={
            "item_location_id": "search_location_id",
            "latitude": "query_latitude",
            "longitude": "query_longitude",
        }
    )[
        [
            "search_location_id",
            "query_latitude",
            "query_longitude",
        ]
    ]
)

In [7]:
fallback_location_centers = (
    train
    .dropna(
        subset=[
            "search_location_id",
            "item_latitude",
            "item_longitude",
        ]
    )
    .groupby(
        "search_location_id",
        as_index=False,
    )
    .agg(
        fallback_latitude=(
            "item_latitude",
            "median",
        ),
        fallback_longitude=(
            "item_longitude",
            "median",
        ),
    )
)

In [8]:
benchmark_queries_with_geo = (
    benchmark_queries
    .merge(
        query_location_centers,
        on="search_location_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        fallback_location_centers,
        on="search_location_id",
        how="left",
        validate="many_to_one",
    )
)

benchmark_queries_with_geo["query_latitude"] = (
    benchmark_queries_with_geo["query_latitude"]
    .fillna(
        benchmark_queries_with_geo[
            "fallback_latitude"
        ]
    )
)

benchmark_queries_with_geo["query_longitude"] = (
    benchmark_queries_with_geo["query_longitude"]
    .fillna(
        benchmark_queries_with_geo[
            "fallback_longitude"
        ]
    )
)

benchmark_queries_with_geo = (
    benchmark_queries_with_geo
    .drop(
        columns=[
            "fallback_latitude",
            "fallback_longitude",
        ]
    )
)

In [9]:
assert np.array_equal(
    benchmark_queries_with_geo["query_id"].to_numpy(),
    benchmark_queries["query_id"].to_numpy(),
)

benchmark_queries_with_geo[
    ["query_latitude", "query_longitude"]
].isna().mean()

query_latitude     0.0
query_longitude    0.0
dtype: float64

In [10]:
item_location_centers = (
    location_centers
    .reset_index()
    .rename(
        columns={
            "latitude": "location_latitude",
            "longitude": "location_longitude",
        }
    )
)

benchmark_items_with_geo = benchmark_items.merge(
    item_location_centers,
    on="item_location_id",
    how="left",
    validate="many_to_one",
)

assert np.array_equal(
    benchmark_items_with_geo["item_id"].to_numpy(),
    benchmark_items["item_id"].to_numpy(),
)

In [11]:
item_latitudes = (
    pd.to_numeric(
        benchmark_items_with_geo["item_latitude"],
        errors="coerce",
    )
    .fillna(
        benchmark_items_with_geo["location_latitude"]
    )
    .to_numpy(dtype=np.float32)
)

item_longitudes = (
    pd.to_numeric(
        benchmark_items_with_geo["item_longitude"],
        errors="coerce",
    )
    .fillna(
        benchmark_items_with_geo["location_longitude"]
    )
    .to_numpy(dtype=np.float32)
)

query_latitudes = pd.to_numeric(
    benchmark_queries_with_geo["query_latitude"],
    errors="coerce",
).to_numpy(dtype=np.float32)

query_longitudes = pd.to_numeric(
    benchmark_queries_with_geo["query_longitude"],
    errors="coerce",
).to_numpy(dtype=np.float32)

In [12]:
pd.Series({
    "item_coordinate_coverage": np.mean(
        np.isfinite(item_latitudes)
        & np.isfinite(item_longitudes)
    ),
    "query_coordinate_coverage": np.mean(
        np.isfinite(query_latitudes)
        & np.isfinite(query_longitudes)
    ),
})

item_coordinate_coverage     1.0
query_coordinate_coverage    1.0
dtype: float64

In [13]:
np.save(
    ARTIFACTS_DIR / "item_ids.npy",
    np.asarray(
        benchmark_items["item_id"],
        dtype="U16",
    ),
)

np.save(
    ARTIFACTS_DIR / "query_ids.npy",
    np.asarray(
        benchmark_queries["query_id"],
        dtype="U16",
    ),
)

np.save(
    ARTIFACTS_DIR / "item_location_ids.npy",
    benchmark_items["item_location_id"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
)

np.save(
    ARTIFACTS_DIR / "item_category_ids.npy",
    benchmark_items["item_category_id"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
)

np.save(
    ARTIFACTS_DIR / "query_location_ids.npy",
    benchmark_queries["search_location_id"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
)

np.save(
    ARTIFACTS_DIR / "query_category_ids.npy",
    benchmark_queries["search_category"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
)

np.save(
    ARTIFACTS_DIR / "item_latitudes.npy",
    item_latitudes,
)
np.save(
    ARTIFACTS_DIR / "item_longitudes.npy",
    item_longitudes,
)
np.save(
    ARTIFACTS_DIR / "query_latitudes.npy",
    query_latitudes,
)
np.save(
    ARTIFACTS_DIR / "query_longitudes.npy",
    query_longitudes,
)

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval import (
    encode_items,
    encode_queries,
    load_semantic_model,
)
from src.text_features import (
    build_query_text,
    build_semantic_item_text,
)

DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "benchmark"

os.environ.pop("SSLKEYLOGFILE", None)

/Users/arina/Documents/Projects/RecSys_candidate_generation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/home/username/sslkeys.log'

In [2]:
benchmark_queries = pd.read_parquet(
    DATA_DIR / "benchmark_queries.parquet"
).reset_index(drop=True)

benchmark_items = pd.read_parquet(
    DATA_DIR / "benchmark_items.parquet"
).reset_index(drop=True)

query_texts = build_query_text(
    benchmark_queries
)

semantic_item_texts = build_semantic_item_text(
    benchmark_items
)

In [3]:
ITEM_EMBEDDINGS_PATH = (
    ARTIFACTS_DIR / "item_embeddings_e5_small.npy"
)

QUERY_EMBEDDINGS_PATH = (
    ARTIFACTS_DIR / "query_embeddings_e5_small.npy"
)

In [4]:
semantic_model = None

if (
    not ITEM_EMBEDDINGS_PATH.exists()
    or not QUERY_EMBEDDINGS_PATH.exists()
):
    semantic_model = load_semantic_model()
    semantic_model.max_seq_length = 256

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9967.36it/s]


In [5]:
if ITEM_EMBEDDINGS_PATH.exists():
    item_embeddings = np.load(
        ITEM_EMBEDDINGS_PATH,
        mmap_mode="r",
    )
else:
    item_embeddings = encode_items(
        model=semantic_model,
        item_texts=semantic_item_texts,
        batch_size=64,
    )

    np.save(
        ITEM_EMBEDDINGS_PATH,
        item_embeddings,
    )

Batches: 100%|██████████| 2957/2957 [28:44<00:00,  1.71it/s]


In [6]:
if QUERY_EMBEDDINGS_PATH.exists():
    query_embeddings = np.load(
        QUERY_EMBEDDINGS_PATH,
        mmap_mode="r",
    )
else:
    query_embeddings = encode_queries(
        model=semantic_model,
        query_texts=query_texts,
        batch_size=128,
    )

    np.save(
        QUERY_EMBEDDINGS_PATH,
        query_embeddings,
    )

Batches: 100%|██████████| 20/20 [00:02<00:00,  8.84it/s]


In [7]:
(
    item_embeddings.shape,
    query_embeddings.shape,
    item_embeddings.dtype,
    query_embeddings.dtype,
)

((189212, 384), (2452, 384), dtype('float32'), dtype('float32'))

In [8]:
(
    np.linalg.norm(item_embeddings[:5], axis=1),
    np.linalg.norm(query_embeddings[:5], axis=1),
)

(array([1., 1., 1., 1., 1.], dtype=float32),
 array([0.99999994, 1.        , 1.        , 1.        , 1.0000001 ],
       dtype=float32))

## Global E5

Для каждого benchmark-запроса ищем 100 семантически ближайших
объявлений во всём корпусе. Для быстрого приближённого поиска
используется индекс HNSW.

In [1]:
from pathlib import Path

import faiss
import numpy as np

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = (
    PROJECT_ROOT / "artifacts" / "benchmark"
)

item_embeddings = np.load(
    ARTIFACTS_DIR / "item_embeddings_e5_small.npy",
    mmap_mode="r",
)

query_embeddings = np.load(
    ARTIFACTS_DIR / "query_embeddings_e5_small.npy",
    mmap_mode="r",
)

item_embeddings.shape, query_embeddings.shape

((189212, 384), (2452, 384))

In [2]:
import gc


GLOBAL_INDEX_PATH = (
    ARTIFACTS_DIR / "global_e5_hnsw_m16.faiss"
)

GLOBAL_INDICES_PATH = (
    ARTIFACTS_DIR / "global_e5_top100_indices.npy"
)

GLOBAL_SCORES_PATH = (
    ARTIFACTS_DIR / "global_e5_top100_scores.npy"
)

global_e5_index = None

if (
    GLOBAL_INDICES_PATH.exists()
    and GLOBAL_SCORES_PATH.exists()
):
    global_e5_indices = np.load(
        GLOBAL_INDICES_PATH,
        mmap_mode="r",
    )

    global_e5_scores = np.load(
        GLOBAL_SCORES_PATH,
        mmap_mode="r",
    )

else:
    faiss.omp_set_num_threads(4)

    if GLOBAL_INDEX_PATH.exists():
        global_e5_index = faiss.read_index(
            str(GLOBAL_INDEX_PATH)
        )

    else:
        global_e5_index = faiss.IndexHNSWFlat(
            item_embeddings.shape[1],
            16,
            faiss.METRIC_INNER_PRODUCT,
        )

        global_e5_index.hnsw.efConstruction = 80

        for start in range(
            0,
            len(item_embeddings),
            5_000,
        ):
            end = min(
                start + 5_000,
                len(item_embeddings),
            )

            global_e5_index.add(
                np.ascontiguousarray(
                    item_embeddings[start:end],
                    dtype=np.float32,
                )
            )

        faiss.write_index(
            global_e5_index,
            str(GLOBAL_INDEX_PATH),
        )

    global_e5_index.hnsw.efSearch = 128

    global_e5_scores, global_e5_indices = (
        global_e5_index.search(
            np.ascontiguousarray(
                query_embeddings,
                dtype=np.float32,
            ),
            100,
        )
    )

    global_e5_indices = global_e5_indices.astype(
        np.int32
    )

    np.save(
        GLOBAL_INDICES_PATH,
        global_e5_indices,
    )

    np.save(
        GLOBAL_SCORES_PATH,
        global_e5_scores,
    )

In [3]:
(
    global_e5_indices.shape,
    global_e5_scores.shape,
)

((2452, 100), (2452, 100))

## Local E5

Для каждого запроса ищем 40 семантически ближайших объявлений
с совпадающими категорией и идентификатором локации.

In [4]:

import pandas as pd

if (
    "global_e5_index" in globals()
    and global_e5_index is not None
):
    del global_e5_index

gc.collect()

0

In [5]:
item_location_ids = np.load(
    ARTIFACTS_DIR / "item_location_ids.npy",
    mmap_mode="r",
)

item_category_ids = np.load(
    ARTIFACTS_DIR / "item_category_ids.npy",
    mmap_mode="r",
)

query_location_ids = np.load(
    ARTIFACTS_DIR / "query_location_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    ARTIFACTS_DIR / "query_category_ids.npy",
    mmap_mode="r",
)

item_groups = pd.DataFrame({
    "location": item_location_ids,
    "category": item_category_ids,
}).groupby(
    ["location", "category"],
    sort=False,
).indices

query_groups = pd.DataFrame({
    "location": query_location_ids,
    "category": query_category_ids,
}).groupby(
    ["location", "category"],
    sort=False,
).indices

len(item_groups), len(query_groups)

(3712, 343)

In [6]:
LOCAL_E5_INDICES_PATH = (
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_indices.npy"
)

LOCAL_E5_SCORES_PATH = (
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_scores.npy"
)

LOCAL_E5_WORKING_INDICES_PATH = (
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_indices.working.npy"
)

LOCAL_E5_WORKING_SCORES_PATH = (
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_scores.working.npy"
)

top_k = 40

if (
    LOCAL_E5_INDICES_PATH.exists()
    and LOCAL_E5_SCORES_PATH.exists()
):
    local_e5_indices = np.load(
        LOCAL_E5_INDICES_PATH,
        mmap_mode="r",
    )

    local_e5_scores = np.load(
        LOCAL_E5_SCORES_PATH,
        mmap_mode="r",
    )

else:
    local_e5_indices = np.lib.format.open_memmap(
        LOCAL_E5_WORKING_INDICES_PATH,
        mode="w+",
        dtype=np.int32,
        shape=(len(query_embeddings), top_k),
    )

    local_e5_scores = np.lib.format.open_memmap(
        LOCAL_E5_WORKING_SCORES_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(len(query_embeddings), top_k),
    )

    local_e5_indices[:] = -1
    local_e5_scores[:] = -np.inf

    faiss.omp_set_num_threads(4)

    for group_number, (
        group_key,
        query_positions,
    ) in enumerate(query_groups.items()):
        item_positions = item_groups.get(
            group_key
        )

        if item_positions is None:
            continue

        item_positions = np.asarray(
            item_positions,
            dtype=np.int64,
        )

        query_positions = np.asarray(
            query_positions,
            dtype=np.int64,
        )

        neighbors_count = min(
            top_k,
            len(item_positions),
        )

        local_index = faiss.IndexFlatIP(
            item_embeddings.shape[1]
        )

        for start in range(
            0,
            len(item_positions),
            5_000,
        ):
            positions = item_positions[
                start:start + 5_000
            ]

            local_index.add(
                np.ascontiguousarray(
                    item_embeddings[positions],
                    dtype=np.float32,
                )
            )

        scores, relative_indices = (
            local_index.search(
                np.ascontiguousarray(
                    query_embeddings[query_positions],
                    dtype=np.float32,
                ),
                neighbors_count,
            )
        )

        local_e5_indices[
            query_positions,
            :neighbors_count,
        ] = item_positions[relative_indices]

        local_e5_scores[
            query_positions,
            :neighbors_count,
        ] = scores

        del local_index

        if group_number % 100 == 0:
            local_e5_indices.flush()
            local_e5_scores.flush()

    local_e5_indices.flush()
    local_e5_scores.flush()

    del local_e5_indices
    del local_e5_scores

    LOCAL_E5_WORKING_INDICES_PATH.replace(
        LOCAL_E5_INDICES_PATH
    )

    LOCAL_E5_WORKING_SCORES_PATH.replace(
        LOCAL_E5_SCORES_PATH
    )

    local_e5_indices = np.load(
        LOCAL_E5_INDICES_PATH,
        mmap_mode="r",
    )

    local_e5_scores = np.load(
        LOCAL_E5_SCORES_PATH,
        mmap_mode="r",
    )

In [7]:
pd.Series({
    "queries": len(query_embeddings),
    "queries_with_local_candidates": (
        local_e5_indices[:, 0] >= 0
    ).sum(),
    "coverage": (
        local_e5_indices[:, 0] >= 0
    ).mean(),
    "mean_candidates": (
        local_e5_indices >= 0
    ).sum(axis=1).mean(),
})

queries                          2452.000000
queries_with_local_candidates    1867.000000
coverage                            0.761419
mean_candidates                    30.440457
dtype: float64

## Local TF-IDF

TF-IDF ищет точные лексические совпадения между текстом запроса
и заголовком с параметрами объявления. Поиск ограничивается той же
локацией и категорией.

In [10]:
import sys

import joblib
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

sys.path.insert(0, str(PROJECT_ROOT))

from src.text_features import (
    build_lexical_item_text,
    build_query_text,
)

DATA_DIR = PROJECT_ROOT / "dataset"

TFIDF_VECTORIZER_PATH = (
    ARTIFACTS_DIR / "tfidf_vectorizer.joblib"
)

TFIDF_ITEM_MATRIX_PATH = (
    ARTIFACTS_DIR / "tfidf_item_matrix.npz"
)

TFIDF_QUERY_MATRIX_PATH = (
    ARTIFACTS_DIR / "tfidf_query_matrix.npz"
)

In [11]:
if (
    TFIDF_VECTORIZER_PATH.exists()
    and TFIDF_ITEM_MATRIX_PATH.exists()
    and TFIDF_QUERY_MATRIX_PATH.exists()
):
    tfidf_vectorizer = joblib.load(
        TFIDF_VECTORIZER_PATH
    )

    tfidf_item_matrix = load_npz(
        TFIDF_ITEM_MATRIX_PATH
    ).tocsr()

    tfidf_query_matrix = load_npz(
        TFIDF_QUERY_MATRIX_PATH
    ).tocsr()

else:
    benchmark_queries = pd.read_parquet(
        DATA_DIR / "benchmark_queries.parquet"
    ).reset_index(drop=True)

    benchmark_items = pd.read_parquet(
        DATA_DIR / "benchmark_items.parquet"
    ).reset_index(drop=True)

    query_texts = build_query_text(
        benchmark_queries
    )

    lexical_item_texts = build_lexical_item_text(
        benchmark_items
    )

    tfidf_vectorizer = TfidfVectorizer(
        lowercase=True,
        dtype=np.float32,
    )

    tfidf_item_matrix = (
        tfidf_vectorizer.fit_transform(
            lexical_item_texts
        )
    )

    tfidf_query_matrix = (
        tfidf_vectorizer.transform(
            query_texts
        )
    )

    joblib.dump(
        tfidf_vectorizer,
        TFIDF_VECTORIZER_PATH,
    )

    save_npz(
        TFIDF_ITEM_MATRIX_PATH,
        tfidf_item_matrix,
    )

    save_npz(
        TFIDF_QUERY_MATRIX_PATH,
        tfidf_query_matrix,
    )

In [12]:
(
    tfidf_item_matrix.shape,
    tfidf_query_matrix.shape,
)

((189212, 127035), (2452, 127035))

In [13]:
LOCAL_TFIDF_INDICES_PATH = (
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy"
)

LOCAL_TFIDF_SCORES_PATH = (
    ARTIFACTS_DIR / "local_tfidf_top40_scores.npy"
)

top_k = 40

if (
    LOCAL_TFIDF_INDICES_PATH.exists()
    and LOCAL_TFIDF_SCORES_PATH.exists()
):
    local_tfidf_indices = np.load(
        LOCAL_TFIDF_INDICES_PATH,
        mmap_mode="r",
    )

    local_tfidf_scores = np.load(
        LOCAL_TFIDF_SCORES_PATH,
        mmap_mode="r",
    )

else:
    local_tfidf_indices = np.lib.format.open_memmap(
        LOCAL_TFIDF_INDICES_PATH,
        mode="w+",
        dtype=np.int32,
        shape=(len(query_location_ids), top_k),
    )

    local_tfidf_scores = np.lib.format.open_memmap(
        LOCAL_TFIDF_SCORES_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(len(query_location_ids), top_k),
    )

    local_tfidf_indices[:] = -1
    local_tfidf_scores[:] = -np.inf

    for group_number, (
        group_key,
        query_positions,
    ) in enumerate(query_groups.items()):
        item_positions = item_groups.get(
            group_key
        )

        if item_positions is None:
            continue

        item_positions = np.asarray(
            item_positions,
            dtype=np.int64,
        )

        query_positions = np.asarray(
            query_positions,
            dtype=np.int64,
        )

        neighbors_count = min(
            top_k,
            len(item_positions),
        )

        search_index = NearestNeighbors(
            metric="cosine",
            algorithm="brute",
            n_jobs=4,
        )

        search_index.fit(
            tfidf_item_matrix[item_positions]
        )

        distances, relative_indices = (
            search_index.kneighbors(
                tfidf_query_matrix[query_positions],
                n_neighbors=neighbors_count,
            )
        )

        local_tfidf_indices[
            query_positions,
            :neighbors_count,
        ] = item_positions[relative_indices]

        local_tfidf_scores[
            query_positions,
            :neighbors_count,
        ] = 1 - distances

        if group_number % 100 == 0:
            local_tfidf_indices.flush()
            local_tfidf_scores.flush()

    local_tfidf_indices.flush()
    local_tfidf_scores.flush()

In [14]:
pd.Series({
    "coverage": (
        local_tfidf_indices[:, 0] >= 0
    ).mean(),
    "mean_candidates": (
        local_tfidf_indices >= 0
    ).sum(axis=1).mean(),
})

coverage            0.761419
mean_candidates    30.440457
dtype: float64

In [15]:
for variable_name in [
    "tfidf_item_matrix",
    "tfidf_query_matrix",
    "tfidf_vectorizer",
    "benchmark_items",
    "benchmark_queries",
    "lexical_item_texts",
    "query_texts",
    "search_index",
]:
    globals().pop(variable_name, None)

gc.collect()

428

## Nearby E5

Канал ищет семантически похожие объявления той же категории,
расположенные в других локациях в радиусе 100 километров.
Он помогает находить релевантные объявления за пределами точного
`search_location_id`.

In [ ]:
import gc


GLOBAL_INDEX_PATH = (
    ARTIFACTS_DIR / "global_e5_hnsw_m16.faiss"
)

GLOBAL_INDICES_PATH = (
    ARTIFACTS_DIR / "global_e5_top100_indices.npy"
)

GLOBAL_SCORES_PATH = (
    ARTIFACTS_DIR / "global_e5_top100_scores.npy"
)

global_e5_index = None

if (
    GLOBAL_INDICES_PATH.exists()
    and GLOBAL_SCORES_PATH.exists()
):
    global_e5_indices = np.load(
        GLOBAL_INDICES_PATH,
        mmap_mode="r",
    )

    global_e5_scores = np.load(
        GLOBAL_SCORES_PATH,
        mmap_mode="r",
    )

else:
    faiss.omp_set_num_threads(4)

    if GLOBAL_INDEX_PATH.exists():
        global_e5_index = faiss.read_index(
            str(GLOBAL_INDEX_PATH)
        )

    else:
        global_e5_index = faiss.IndexHNSWFlat(
            item_embeddings.shape[1],
            16,
            faiss.METRIC_INNER_PRODUCT,
        )

        global_e5_index.hnsw.efConstruction = 80

        for start in range(
            0,
            len(item_embeddings),
            5_000,
        ):
            end = min(
                start + 5_000,
                len(item_embeddings),
            )

            global_e5_index.add(
                np.ascontiguousarray(
                    item_embeddings[start:end],
                    dtype=np.float32,
                )
            )

        faiss.write_index(
            global_e5_index,
            str(GLOBAL_INDEX_PATH),
        )

    global_e5_index.hnsw.efSearch = 128

    global_e5_scores, global_e5_indices = (
        global_e5_index.search(
            np.ascontiguousarray(
                query_embeddings,
                dtype=np.float32,
            ),
            100,
        )
    )

    global_e5_indices = global_e5_indices.astype(
        np.int32
    )

    np.save(
        GLOBAL_INDICES_PATH,
        global_e5_indices,
    )

    np.save(
        GLOBAL_SCORES_PATH,
        global_e5_scores,
    )

In [16]:
from sklearn.neighbors import BallTree

item_latitudes = np.load(
    ARTIFACTS_DIR / "item_latitudes.npy",
    mmap_mode="r",
)

item_longitudes = np.load(
    ARTIFACTS_DIR / "item_longitudes.npy",
    mmap_mode="r",
)

query_latitudes = np.load(
    ARTIFACTS_DIR / "query_latitudes.npy",
    mmap_mode="r",
)

query_longitudes = np.load(
    ARTIFACTS_DIR / "query_longitudes.npy",
    mmap_mode="r",
)

valid_item_mask = (
    np.isfinite(item_latitudes)
    & np.isfinite(item_longitudes)
)

valid_item_positions = np.flatnonzero(
    valid_item_mask
)

item_coordinates_radians = np.radians(
    np.column_stack([
        item_latitudes[valid_item_positions],
        item_longitudes[valid_item_positions],
    ])
)

geo_index = BallTree(
    item_coordinates_radians,
    metric="haversine",
    leaf_size=40,
)

len(valid_item_positions), len(item_latitudes)

(189212, 189212)

In [17]:
NEARBY_E5_INDICES_PATH = (
    ARTIFACTS_DIR / "nearby_semantic_top20_indices.npy"
)

NEARBY_E5_WORKING_PATH = (
    ARTIFACTS_DIR
    / "nearby_semantic_top20_indices.working.npy"
)

nearby_top_k = 20
earth_radius_km = 6371.0
radius_km = 100.0
radius_radians = radius_km / earth_radius_km

if NEARBY_E5_INDICES_PATH.exists():
    nearby_e5_indices = np.load(
        NEARBY_E5_INDICES_PATH,
        mmap_mode="r",
    )

else:
    nearby_e5_indices = np.lib.format.open_memmap(
        NEARBY_E5_WORKING_PATH,
        mode="w+",
        dtype=np.int32,
        shape=(len(query_embeddings), nearby_top_k),
    )

    nearby_e5_indices[:] = -1

    faiss.omp_set_num_threads(4)

    for group_number, (
        group_key,
        query_positions,
    ) in enumerate(query_groups.items()):
        location_id, category_id = group_key

        query_positions = np.asarray(
            query_positions,
            dtype=np.int64,
        )

        finite_coordinates = (
            np.isfinite(
                query_latitudes[query_positions]
            )
            & np.isfinite(
                query_longitudes[query_positions]
            )
        )

        if not finite_coordinates.any():
            continue

        coordinate_positions = query_positions[
            finite_coordinates
        ]

        query_center_radians = np.radians([[
            np.nanmedian(
                query_latitudes[coordinate_positions]
            ),
            np.nanmedian(
                query_longitudes[coordinate_positions]
            ),
        ]])

        relative_positions = geo_index.query_radius(
            query_center_radians,
            r=radius_radians,
        )[0]

        candidate_positions = valid_item_positions[
            relative_positions
        ]

        candidate_positions = candidate_positions[
            (
                item_category_ids[candidate_positions]
                == category_id
            )
            & (
                item_location_ids[candidate_positions]
                != location_id
            )
        ]

        if len(candidate_positions) == 0:
            continue

        neighbors_count = min(
            nearby_top_k,
            len(candidate_positions),
        )

        nearby_index = faiss.IndexFlatIP(
            item_embeddings.shape[1]
        )

        for start in range(
            0,
            len(candidate_positions),
            5_000,
        ):
            positions = candidate_positions[
                start:start + 5_000
            ]

            nearby_index.add(
                np.ascontiguousarray(
                    item_embeddings[positions],
                    dtype=np.float32,
                )
            )

        scores, relative_indices = (
            nearby_index.search(
                np.ascontiguousarray(
                    query_embeddings[query_positions],
                    dtype=np.float32,
                ),
                neighbors_count,
            )
        )

        nearby_e5_indices[
            query_positions,
            :neighbors_count,
        ] = candidate_positions[
            relative_indices
        ]

        del nearby_index

        if group_number % 100 == 0:
            nearby_e5_indices.flush()

            print(
                f"Обработано {group_number} "
                f"из {len(query_groups)} групп"
            )

    nearby_e5_indices.flush()
    del nearby_e5_indices

    NEARBY_E5_WORKING_PATH.replace(
        NEARBY_E5_INDICES_PATH
    )

    nearby_e5_indices = np.load(
        NEARBY_E5_INDICES_PATH,
        mmap_mode="r",
    )

Обработано 0 из 343 групп
Обработано 100 из 343 групп
Обработано 300 из 343 групп


In [18]:
nearby_counts = (
    nearby_e5_indices >= 0
).sum(axis=1)

pd.Series({
    "coverage": (
        nearby_counts > 0
    ).mean(),
    "mean_candidates": nearby_counts.mean(),
    "median_candidates": np.median(
        nearby_counts
    ),
    "max_candidates": nearby_counts.max(),
})

coverage              0.909054
mean_candidates      18.025285
median_candidates    20.000000
max_candidates       20.000000
dtype: float64

## Обучение финального агрегатора

После выбора Logistic Regression переобучаем модель на всех
локальных validation-запросах. Затем сохраняем весь pipeline,
включая StandardScaler и Logistic Regression.

In [1]:
from pathlib import Path
import gc

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

In [2]:
train_pairs = pd.read_parquet(
    DATA_DIR / "train.parquet",
    columns=SEARCH_COLUMNS + ["item_id"],
)

query_group_ids = train_pairs.groupby(
    SEARCH_COLUMNS,
    dropna=False,
).ngroup()

unique_group_ids = query_group_ids.unique()

_, validation_group_ids = train_test_split(
    unique_group_ids,
    test_size=0.2,
    random_state=42,
)

validation_rows = train_pairs[
    query_group_ids.isin(validation_group_ids)
]

validation_queries = (
    validation_rows
    .drop_duplicates(
        SEARCH_COLUMNS + ["item_id"]
    )
    .groupby(
        SEARCH_COLUMNS,
        dropna=False,
    )
    .agg(
        relevant_item_ids=("item_id", list)
    )
    .reset_index()
)

validation_queries.shape

(70893, 6)

In [3]:
saved_location_ids = np.load(
    ARTIFACTS_DIR / "validation_location_ids.npy",
    mmap_mode="r",
)

saved_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

assert np.array_equal(
    validation_queries["search_location_id"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
    saved_location_ids,
)

assert np.array_equal(
    validation_queries["search_category"]
    .fillna(-1)
    .astype(np.int64)
    .to_numpy(),
    saved_category_ids,
)

print("Validation восстановлена корректно")

Validation восстановлена корректно


In [4]:
local_e5 = np.load(
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_indices.npy",
    mmap_mode="r",
)

local_e5_scores = np.load(
    ARTIFACTS_DIR
    / "local_semantic_exact_top40_scores.npy",
    mmap_mode="r",
)

local_tfidf = np.load(
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy",
    mmap_mode="r",
)

local_tfidf_scores = np.load(
    ARTIFACTS_DIR / "local_tfidf_top40_scores.npy",
    mmap_mode="r",
)

nearby_e5 = np.load(
    ARTIFACTS_DIR
    / "nearby_semantic_top20_indices.npy",
    mmap_mode="r",
)

global_e5 = np.load(
    ARTIFACTS_DIR
    / "local_semantic_top100_indices.npy",
    mmap_mode="r",
)

global_e5_scores = np.load(
    ARTIFACTS_DIR
    / "local_semantic_top100_scores.npy",
    mmap_mode="r",
)

In [5]:
candidate_item_ids = np.load(
    ARTIFACTS_DIR / "candidate_item_ids.npy",
    allow_pickle=True,
)

candidate_location_ids = np.load(
    ARTIFACTS_DIR / "candidate_location_ids.npy",
    mmap_mode="r",
)

candidate_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)

query_location_ids = np.load(
    ARTIFACTS_DIR / "validation_location_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [6]:
assert local_e5.shape == (70893, 40)
assert local_tfidf.shape == (70893, 40)
assert nearby_e5.shape == (70893, 20)
assert global_e5.shape == (70893, 100)

assert len(candidate_item_ids) == 344825
assert len(query_location_ids) == 70893

print("Локальные retrieval-артефакты загружены")

Локальные retrieval-артефакты загружены


In [7]:
FEATURE_NAMES = [
    "local_e5_reciprocal_rank",
    "local_tfidf_reciprocal_rank",
    "nearby_e5_reciprocal_rank",
    "global_e5_reciprocal_rank",
    "in_local_e5",
    "in_local_tfidf",
    "in_nearby_e5",
    "in_global_e5",
    "local_e5_score",
    "local_tfidf_score",
    "global_e5_score",
    "channels_count",
    "same_location",
    "same_category",
]

In [8]:
def build_query_candidate_features(
    query_position,
):
    features_by_item = {}

    channel_specs = [
        (local_e5, local_e5_scores, 8),
        (local_tfidf, local_tfidf_scores, 9),
        (nearby_e5, None, None),
        (global_e5, global_e5_scores, 10),
    ]

    for channel_number, (
        indices,
        scores,
        score_column,
    ) in enumerate(channel_specs):
        for rank, item_position in enumerate(
            indices[query_position],
            start=1,
        ):
            if item_position < 0:
                continue

            item_position = int(item_position)

            if item_position not in features_by_item:
                features_by_item[item_position] = (
                    np.zeros(
                        len(FEATURE_NAMES),
                        dtype=np.float32,
                    )
                )

            features = features_by_item[
                item_position
            ]

            features[channel_number] = (
                1.0 / rank
            )

            features[
                4 + channel_number
            ] = 1.0

            if score_column is not None:
                score = scores[
                    query_position,
                    rank - 1,
                ]

                if np.isfinite(score):
                    features[
                        score_column
                    ] = score

    item_positions = np.fromiter(
        features_by_item.keys(),
        dtype=np.int32,
    )

    feature_matrix = np.vstack(
        list(features_by_item.values())
    )

    feature_matrix[:, 11] = (
        feature_matrix[:, 4:8].sum(axis=1)
    )

    feature_matrix[:, 12] = (
        candidate_location_ids[item_positions]
        == query_location_ids[query_position]
    )

    feature_matrix[:, 13] = (
        candidate_category_ids[item_positions]
        == query_category_ids[query_position]
    )

    return item_positions, feature_matrix

In [9]:
example_positions, example_features = (
    build_query_candidate_features(0)
)

(
    example_positions.shape,
    example_features.shape,
    np.isfinite(example_features).all(),
)

((179,), (179, 14), np.True_)

In [10]:
for variable_name in [
    "train_pairs",
    "validation_rows",
    "query_group_ids",
    "unique_group_ids",
    "validation_group_ids",
    "saved_location_ids",
    "saved_category_ids",
]:
    globals().pop(variable_name, None)

gc.collect()

0

In [11]:
rng = np.random.default_rng(42)

final_feature_parts = []
final_label_parts = []

queries_with_positive = 0

for query_position in range(
    len(validation_queries)
):
    item_positions, features = (
        build_query_candidate_features(
            query_position
        )
    )

    relevant_items = set(
        validation_queries.iloc[
            query_position
        ]["relevant_item_ids"]
    )

    labels = np.fromiter(
        (
            candidate_item_ids[item_position]
            in relevant_items
            for item_position in item_positions
        ),
        dtype=np.int8,
    )

    positive_rows = np.flatnonzero(
        labels == 1
    )

    negative_rows = np.flatnonzero(
        labels == 0
    )

    # Если retrieval не нашёл ни одного
    # релевантного объявления, агрегатору
    # не на чем учиться для этого запроса.
    if len(positive_rows) == 0:
        continue

    queries_with_positive += 1

    number_of_negatives = min(
        20,
        len(negative_rows),
    )

    sampled_negative_rows = rng.choice(
        negative_rows,
        size=number_of_negatives,
        replace=False,
    )

    selected_rows = np.concatenate([
        positive_rows,
        sampled_negative_rows,
    ])

    final_feature_parts.append(
        features[selected_rows]
    )

    final_label_parts.append(
        labels[selected_rows]
    )

    if query_position % 5_000 == 0:
        print(
            f"Обработано {query_position} "
            f"из {len(validation_queries)} запросов"
        )

Обработано 0 из 70893 запросов
Обработано 5000 из 70893 запросов
Обработано 15000 из 70893 запросов
Обработано 20000 из 70893 запросов
Обработано 25000 из 70893 запросов
Обработано 30000 из 70893 запросов
Обработано 35000 из 70893 запросов
Обработано 40000 из 70893 запросов
Обработано 50000 из 70893 запросов
Обработано 55000 из 70893 запросов
Обработано 60000 из 70893 запросов
Обработано 65000 из 70893 запросов
Обработано 70000 из 70893 запросов


In [12]:
X_final = np.vstack(
    final_feature_parts
).astype(np.float32)

y_final = np.concatenate(
    final_label_parts
)

del final_feature_parts
del final_label_parts

gc.collect()

pd.Series({
    "training_pairs": len(y_final),
    "features": X_final.shape[1],
    "positive_share": y_final.mean(),
    "queries_with_positive": (
        queries_with_positive
    ),
    "query_pool_coverage": (
        queries_with_positive
        / len(validation_queries)
    ),
})

training_pairs           1.291806e+06
features                 1.400000e+01
positive_share           5.988980e-02
queries_with_positive    6.072200e+04
query_pool_coverage      8.565303e-01
dtype: float64

In [13]:
final_linear_blender = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=500,
        random_state=42,
    ),
)

final_linear_blender.fit(
    X_final,
    y_final,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [14]:
FINAL_MODEL_PATH = (
    ARTIFACTS_DIR
    / "final_linear_blender.joblib"
)

final_model_bundle = {
    "model": final_linear_blender,
    "feature_names": FEATURE_NAMES,
}

joblib.dump(
    final_model_bundle,
    FINAL_MODEL_PATH,
)

FINAL_MODEL_PATH, FINAL_MODEL_PATH.stat().st_size

(PosixPath('/Users/arina/Documents/Projects/RecSys_candidate_generation/artifacts/final_linear_blender.joblib'),
 2088)

In [15]:
loaded_bundle = joblib.load(
    FINAL_MODEL_PATH
)

assert (
    loaded_bundle["feature_names"]
    == FEATURE_NAMES
)

assert np.allclose(
    loaded_bundle["model"].predict_proba(
        X_final[:10]
    ),
    final_linear_blender.predict_proba(
        X_final[:10]
    ),
)

print("Финальная модель сохранена корректно")

Финальная модель сохранена корректно


## Финальная агрегация

Загружаем четыре retrieval-канала и обученный pipeline
`StandardScaler + LogisticRegression`. Для каждого объявления из
объединённого пула строим те же 14 признаков, что и при обучении.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

BENCHMARK_ARTIFACTS_DIR = (
    ARTIFACTS_DIR / "benchmark"
)

model_bundle = joblib.load(
    ARTIFACTS_DIR
    / "final_linear_blender.joblib"
)

final_model = model_bundle["model"]
FEATURE_NAMES = model_bundle["feature_names"]

len(FEATURE_NAMES), FEATURE_NAMES

(14,
 ['local_e5_reciprocal_rank',
  'local_tfidf_reciprocal_rank',
  'nearby_e5_reciprocal_rank',
  'global_e5_reciprocal_rank',
  'in_local_e5',
  'in_local_tfidf',
  'in_nearby_e5',
  'in_global_e5',
  'local_e5_score',
  'local_tfidf_score',
  'global_e5_score',
  'channels_count',
  'same_location',
  'same_category'])

In [2]:
local_e5 = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "local_semantic_exact_top40_indices.npy",
    mmap_mode="r",
)

local_e5_scores = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "local_semantic_exact_top40_scores.npy",
    mmap_mode="r",
)

local_tfidf = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "local_tfidf_top40_indices.npy",
    mmap_mode="r",
)

local_tfidf_scores = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "local_tfidf_top40_scores.npy",
    mmap_mode="r",
)

nearby_e5 = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "nearby_semantic_top20_indices.npy",
    mmap_mode="r",
)

global_e5 = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "global_e5_top100_indices.npy",
    mmap_mode="r",
)

global_e5_scores = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "global_e5_top100_scores.npy",
    mmap_mode="r",
)

In [3]:
item_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR / "item_ids.npy",
    mmap_mode="r",
)

query_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR / "query_ids.npy",
    mmap_mode="r",
)

item_location_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "item_location_ids.npy",
    mmap_mode="r",
)

item_category_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "item_category_ids.npy",
    mmap_mode="r",
)

query_location_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "query_location_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    BENCHMARK_ARTIFACTS_DIR
    / "query_category_ids.npy",
    mmap_mode="r",
)

In [4]:
assert local_e5.shape == (2452, 40)
assert local_tfidf.shape == (2452, 40)
assert nearby_e5.shape == (2452, 20)
assert global_e5.shape == (2452, 100)

assert len(item_ids) == 189212
assert len(query_ids) == 2452

print("Все benchmark-артефакты загружены")

Все benchmark-артефакты загружены


In [5]:
def build_benchmark_candidate_features(
    query_position,
):
    features_by_item = {}

    channel_specs = [
        (local_e5, local_e5_scores, 8),
        (local_tfidf, local_tfidf_scores, 9),
        (nearby_e5, None, None),
        (global_e5, global_e5_scores, 10),
    ]

    for channel_number, (
        indices,
        scores,
        score_column,
    ) in enumerate(channel_specs):
        for rank, item_position in enumerate(
            indices[query_position],
            start=1,
        ):
            if item_position < 0:
                continue

            item_position = int(item_position)

            if item_position not in features_by_item:
                features_by_item[item_position] = (
                    np.zeros(
                        len(FEATURE_NAMES),
                        dtype=np.float32,
                    )
                )

            features = features_by_item[
                item_position
            ]

            features[channel_number] = (
                1.0 / rank
            )

            features[
                4 + channel_number
            ] = 1.0

            if score_column is not None:
                score = scores[
                    query_position,
                    rank - 1,
                ]

                if np.isfinite(score):
                    features[
                        score_column
                    ] = score

    candidate_positions = np.fromiter(
        features_by_item.keys(),
        dtype=np.int32,
    )

    feature_matrix = np.vstack(
        list(features_by_item.values())
    )

    feature_matrix[:, 11] = (
        feature_matrix[:, 4:8].sum(axis=1)
    )

    feature_matrix[:, 12] = (
        item_location_ids[candidate_positions]
        == query_location_ids[query_position]
    )

    feature_matrix[:, 13] = (
        item_category_ids[candidate_positions]
        == query_category_ids[query_position]
    )

    return candidate_positions, feature_matrix

In [6]:
def build_benchmark_top50(
    model,
    batch_size=500,
):
    result = np.full(
        (len(query_ids), 50),
        -1,
        dtype=np.int32,
    )

    for batch_start in range(
        0,
        len(query_ids),
        batch_size,
    ):
        batch_query_positions = np.arange(
            batch_start,
            min(
                batch_start + batch_size,
                len(query_ids),
            ),
        )

        batch_features = []
        batch_candidate_positions = []
        batch_lengths = []

        for query_position in batch_query_positions:
            candidate_positions, features = (
                build_benchmark_candidate_features(
                    query_position
                )
            )

            batch_features.append(features)
            batch_candidate_positions.append(
                candidate_positions
            )
            batch_lengths.append(
                len(candidate_positions)
            )

        combined_features = np.vstack(
            batch_features
        )

        probabilities = model.predict_proba(
            combined_features
        )[:, 1]

        offset = 0

        for (
            query_position,
            candidate_positions,
            number_of_candidates,
        ) in zip(
            batch_query_positions,
            batch_candidate_positions,
            batch_lengths,
        ):
            query_probabilities = probabilities[
                offset:offset + number_of_candidates
            ]

            offset += number_of_candidates

            ranking_order = np.argsort(
                -query_probabilities,
                kind="stable",
            )

            ranked_positions = candidate_positions[
                ranking_order
            ]

            same_category = ranked_positions[
                item_category_ids[ranked_positions]
                == query_category_ids[query_position]
            ]

            other_categories = ranked_positions[
                item_category_ids[ranked_positions]
                != query_category_ids[query_position]
            ]

            selected = np.concatenate([
                same_category,
                other_categories,
            ])[:50]

            result[
                query_position,
                :len(selected),
            ] = selected

    return result

In [7]:
benchmark_top50_positions = (
    build_benchmark_top50(
        model=final_model,
        batch_size=500,
    )
)

In [8]:
candidate_counts = (
    benchmark_top50_positions >= 0
).sum(axis=1)

pd.Series(candidate_counts).describe()

count    2452.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64

## Формирование answer.csv

Позиции строк корпуса преобразуются в исходные `item_id`.
Идентификаторы сохраняются без изменения регистра и объединяются
одним пробелом.

In [9]:
predictions = []

for position_row in benchmark_top50_positions:
    top50_item_ids = []
    seen_item_ids = set()

    for item_position in position_row:
        if item_position < 0:
            continue

        item_id = str(
            item_ids[item_position]
        )

        # Дополнительная защита от повторов.
        if item_id not in seen_item_ids:
            top50_item_ids.append(item_id)
            seen_item_ids.add(item_id)

        if len(top50_item_ids) == 50:
            break

    predictions.append(top50_item_ids)

In [10]:
prediction_lengths = pd.Series(
    [len(items) for items in predictions]
)

prediction_lengths.describe()

count    2452.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64

In [11]:
answer = pd.DataFrame({
    "query_id": [
        str(query_id)
        for query_id in query_ids
    ],
    "answer": [
        " ".join(top50)
        for top50 in predictions
    ],
})

answer.shape, answer.head(3)

((2452, 2),
            query_id                                             answer
 0  70DfDUpwjxB4lzFd  355392014208b7bf 603623b4bd9e8f6c d722bcda1a55...
 1  JTrdTaZJvSiLPkXj  422d3ffdd5bbf626 14fbc6d1a2f9f588 f50963ef7dda...
 2  LZCZNoVG4AFUkVRJ  dab52187b4500d9b 168a9207e80b0be4 3f89b8062dc8...)

In [12]:
ANSWER_PATH = PROJECT_ROOT / "answer.csv"

answer.to_csv(
    ANSWER_PATH,
    index=False,
    encoding="utf-8",
)

ANSWER_PATH

PosixPath('/Users/arina/Documents/Projects/RecSys_candidate_generation/answer.csv')

In [13]:
saved_answer = pd.read_csv(
    ANSWER_PATH,
    dtype={
        "query_id": "string",
        "answer": "string",
    },
    keep_default_na=False,
)

saved_predictions = saved_answer[
    "answer"
].map(
    lambda value: (
        value.split(" ")
        if value
        else []
    )
)

In [14]:
assert saved_answer.columns.tolist() == [
    "query_id",
    "answer",
]

assert len(saved_answer) == len(query_ids)
assert saved_answer["query_id"].is_unique

assert set(saved_answer["query_id"]) == set(
    map(str, query_ids)
)

assert saved_answer[
    "query_id"
].str.len().eq(16).all()

assert saved_predictions.map(
    len
).le(50).all()

assert saved_predictions.map(
    lambda items: len(items) == len(set(items))
).all()

assert saved_answer["answer"].map(
    lambda value: value == value.strip()
).all()

assert saved_answer["answer"].map(
    lambda value: "  " not in value
).all()

In [15]:
all_predicted_item_ids = [
    item_id
    for items in saved_predictions
    for item_id in items
]

benchmark_item_id_set = set(
    map(str, item_ids)
)

assert all(
    len(item_id) == 16
    for item_id in all_predicted_item_ids
)

assert all(
    set(item_id) <= set("0123456789abcdef")
    for item_id in all_predicted_item_ids
)

assert set(
    all_predicted_item_ids
).issubset(
    benchmark_item_id_set
)

print("answer.csv прошёл все проверки")

answer.csv прошёл все проверки


In [16]:
pd.Series({
    "rows": len(saved_answer),
    "unique_query_ids": (
        saved_answer["query_id"].nunique()
    ),
    "min_items": saved_predictions.map(len).min(),
    "max_items": saved_predictions.map(len).max(),
    "unique_used_items": len(
        set(all_predicted_item_ids)
    ),
    "file_size_kb": (
        ANSWER_PATH.stat().st_size / 1024
    ),
})

rows                  2452.000000
unique_query_ids      2452.000000
min_items               50.000000
max_items               50.000000
unique_used_items    81652.000000
file_size_kb          2076.074219
dtype: float64

## Альтернативная агрегация: Weighted RRF

Weighted Reciprocal Rank Fusion объединяет ранги четырёх retrieval-каналов,
не используя абсолютные similarity-score. Конфигурация была выбрана
на локальной tuning-выборке.

In [17]:
RRF_CHANNELS = [
    local_e5,
    local_tfidf,
    nearby_e5,
    global_e5,
]

RRF_WEIGHTS = (
    3.0,   # Local E5
    0.5,   # Local TF-IDF
    1.5,   # Nearby E5
    0.5,   # Global E5
)

RRF_K = 30

In [18]:
def build_benchmark_rrf_top50():
    result = np.full(
        (len(query_ids), 50),
        -1,
        dtype=np.int32,
    )

    for query_position in range(
        len(query_ids)
    ):
        rrf_scores = {}

        for channel, weight in zip(
            RRF_CHANNELS,
            RRF_WEIGHTS,
        ):
            for rank, item_position in enumerate(
                channel[query_position],
                start=1,
            ):
                if item_position < 0:
                    continue

                item_position = int(item_position)

                rrf_scores[item_position] = (
                    rrf_scores.get(
                        item_position,
                        0.0,
                    )
                    + weight / (RRF_K + rank)
                )

        ranked_positions = np.asarray(
            sorted(
                rrf_scores,
                key=rrf_scores.get,
                reverse=True,
            ),
            dtype=np.int32,
        )

        # Повторяем тот же category post-filter,
        # который использовался локально.
        same_category = ranked_positions[
            item_category_ids[ranked_positions]
            == query_category_ids[query_position]
        ]

        other_categories = ranked_positions[
            item_category_ids[ranked_positions]
            != query_category_ids[query_position]
        ]

        selected = np.concatenate([
            same_category,
            other_categories,
        ])[:50]

        result[
            query_position,
            :len(selected),
        ] = selected

    return result

In [19]:
rrf_top50_positions = (
    build_benchmark_rrf_top50()
)

In [20]:
rrf_counts = (
    rrf_top50_positions >= 0
).sum(axis=1)

pd.Series(rrf_counts).describe()

count    2452.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64

In [21]:
top50_overlap = []

for logistic_row, rrf_row in zip(
    benchmark_top50_positions,
    rrf_top50_positions,
):
    logistic_items = set(
        logistic_row[logistic_row >= 0]
    )

    rrf_items = set(
        rrf_row[rrf_row >= 0]
    )

    top50_overlap.append(
        len(logistic_items & rrf_items)
    )

pd.Series(top50_overlap).describe()

count    2452.000000
mean       46.454323
std         4.365674
min        19.000000
25%        44.000000
50%        49.000000
75%        50.000000
max        50.000000
dtype: float64

In [22]:
import shutil

SUBMISSIONS_DIR = (
    PROJECT_ROOT / "submissions"
)

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOGREG_ARCHIVE_PATH = (
    SUBMISSIONS_DIR
    / "answer_v1_logreg_0.722516.csv"
)

if not LOGREG_ARCHIVE_PATH.exists():
    shutil.copy2(
        PROJECT_ROOT / "answer.csv",
        LOGREG_ARCHIVE_PATH,
    )

In [23]:
rrf_predictions = []

for position_row in rrf_top50_positions:
    top50_item_ids = []
    seen_item_ids = set()

    for item_position in position_row:
        if item_position < 0:
            continue

        item_id = str(
            item_ids[item_position]
        )

        if item_id not in seen_item_ids:
            top50_item_ids.append(item_id)
            seen_item_ids.add(item_id)

        if len(top50_item_ids) == 50:
            break

    rrf_predictions.append(
        top50_item_ids
    )

rrf_answer = pd.DataFrame({
    "query_id": list(map(str, query_ids)),
    "answer": [
        " ".join(items)
        for items in rrf_predictions
    ],
})

In [24]:
RRF_ARCHIVE_PATH = (
    SUBMISSIONS_DIR
    / "answer_v2_weighted_rrf.csv"
)

rrf_answer.to_csv(
    RRF_ARCHIVE_PATH,
    index=False,
    encoding="utf-8",
)

In [25]:
ANSWER_PATH = PROJECT_ROOT / "answer.csv"

rrf_answer.to_csv(
    ANSWER_PATH,
    index=False,
    encoding="utf-8",
)

In [26]:
saved_rrf_answer = pd.read_csv(
    ANSWER_PATH,
    dtype={
        "query_id": "string",
        "answer": "string",
    },
    keep_default_na=False,
)

parsed_rrf_answers = saved_rrf_answer[
    "answer"
].map(
    lambda value: value.split(" ")
    if value
    else []
)

assert saved_rrf_answer.columns.tolist() == [
    "query_id",
    "answer",
]

assert len(saved_rrf_answer) == 2452
assert saved_rrf_answer["query_id"].is_unique

assert set(saved_rrf_answer["query_id"]) == set(
    map(str, query_ids)
)

assert parsed_rrf_answers.map(len).le(50).all()

assert parsed_rrf_answers.map(
    lambda items: len(items) == len(set(items))
).all()

all_rrf_item_ids = [
    item_id
    for items in parsed_rrf_answers
    for item_id in items
]

assert all(
    len(item_id) == 16
    and set(item_id) <= set("0123456789abcdef")
    for item_id in all_rrf_item_ids
)

assert set(all_rrf_item_ids).issubset(
    set(map(str, item_ids))
)

print("Weighted RRF answer.csv прошёл все проверки")

Weighted RRF answer.csv прошёл все проверки
